# Chapter 2:数据集准备

## 2.2 Tokenizing text

In [13]:
import os
import urllib.request

if not os.path.exists("the-verdict.txt"):
    url=("https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/refs/heads/main/ch02/01_main-chapter-code/the-verdict.txt")
    file_path="the-verdict.txt"
    urllib.request.urlretrieve(url,file_path)


In [14]:
with open("the-verdict.txt","r",encoding="utf-8") as f:
    raw_text=f.read()

In [15]:
raw_text

'I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no great surprise to me to hear that, in the height of his glory, he had dropped his painting, married a rich widow, and established himself in a villa on the Riviera. (Though I rather thought it would have been Rome or Florence.)\n\n"The height of his glory"--that was what the women called it. I can hear Mrs. Gideon Thwing--his last Chicago sitter--deploring his unaccountable abdication. "Of course it\'s going to send the value of my picture \'way up; but I don\'t think of that, Mr. Rickham--the loss to Arrt is all I think of." The word, on Mrs. Thwing\'s lips, multiplied its _rs_ as though they were reflected in an endless vista of mirrors. And it was not only the Mrs. Thwings who mourned. Had not the exquisite Hermia Croft, at the last Grafton Gallery show, stopped me before Gisburn\'s "Moon-dancers" to say, with tears in her eyes: "We shall not look upon its like again"?\n\nWell!--even 

In [16]:
len(raw_text)

20479

In [17]:
import re

text = "Hello, world. This, is a test."
result = re.split(r'(\s)',text)

print(result)

['Hello,', ' ', 'world.', ' ', 'This,', ' ', 'is', ' ', 'a', ' ', 'test.']


In [18]:
result = re.split(r'([,.]|\s)',text)

In [19]:
print(result)

['Hello', ',', '', ' ', 'world', '.', '', ' ', 'This', ',', '', ' ', 'is', ' ', 'a', ' ', 'test', '.', '']


In [20]:
result=[item for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'This', ',', 'is', 'a', 'test', '.']


In [9]:
text = "Hello, world. Is this-- a test?"
result=re.split(r'([,.:;?_!"()\']|--|\s)',raw_text)
result=[item.strip() for item in result if item.strip()]
preprocessed=result

In [10]:
len(preprocessed)

4690

In [11]:
preprocessed[:10]

['I',
 'HAD',
 'always',
 'thought',
 'Jack',
 'Gisburn',
 'rather',
 'a',
 'cheap',
 'genius']

## 2.3 将Tokens转换为IDs（Converting tokens into token IDs)

In [25]:
vocab["Jack"]

57

In [26]:
int_to_str={i:s for s,i in vocab.items()}

int_to_str[57]

'Jack'

In [23]:
all_words=sorted(set(preprocessed))
vocab_size=len(all_words)
print(vocab_size)

1130


In [24]:
vocab={token:integer for integer ,token in enumerate(all_words)}
# vocab

In [27]:
class SimpleTokenizerV1:
    def __init__(self,vocab):
        self.str_to_int=vocab
        self.int_to_str={i:s for s,i in vocab.items()}

    def encode(self,text):
        preprocessed=re.split(r'([,.:;?_!"()\']|--|\s)',text)

        preprocessed=[
            item.strip() for item in preprocessed if item.strip()
        ]
        ids=[self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self,ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces beforethe specified punctuations
        text=re.sub(r'\s+([,.?!"()\'])',r'\1',text)
        return text

In [28]:
tokenizer=SimpleTokenizerV1(vocab)

In [29]:
text="""It's the last he painted, you know,"
        Mrs. Gisburn said with pardonable pride."""

In [30]:
ids=tokenizer.encode(text)
print(ids)

[56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]


In [31]:
tokenizer.decode(ids)

'It\' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.'

In [32]:
tokenizer.decode(ids=tokenizer.encode(text))

'It\' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.'

## 2.4 Adding special context tokens

In [33]:
text="Hello, do you like tea. is this--a test?"
tokenizer.encode(text)

KeyError: 'Hello'

In [34]:
all_tokens=sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>","<|unk|>"])

vocab={token:integer for integer ,token in enumerate(all_tokens)}

In [35]:
len(vocab.items())

1132

In [36]:
for i,item in enumerate(list(vocab.items())[-5:]):
    print(item)

('younger', 1127)
('your', 1128)
('yourself', 1129)
('<|endoftext|>', 1130)
('<|unk|>', 1131)


In [37]:
class SimpleTokenizerV1:
    def __init__(self,vocab):
        self.str_to_int=vocab
        self.int_to_str={i:s for s,i in vocab.items()}

    def encode(self,text):
        preprocessed=re.split(r'([,.:;?_!"()\']|--|\s)',text)

        preprocessed=[
            item.strip() for item in preprocessed if item.strip()
        ]
        preprocessed=[
            item if item in self.str_to_int
            else "<|unk|>" for item in preprocessed
        ]
        ids=[self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self,ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces beforethe specified punctuations
        text=re.sub(r'\s+([,.?!"()\'])',r'\1',text)
        return text

In [38]:
tokenizer=SimpleTokenizerV1(vocab)

In [39]:
tokenizer.encode(text)

[1131, 5, 355, 1126, 628, 975, 7, 584, 999, 6, 115, 1131, 10]

In [40]:
text

'Hello, do you like tea. is this--a test?'

In [41]:
tokenizer.decode(tokenizer.encode(text))

'<|unk|>, do you like tea. is this -- a <|unk|>?'

## 2.5 Byte pair encoding

In [42]:
import tiktoken

In [43]:
# !pip install tiktoken

In [44]:
tiktoken.__version__

'0.12.0'

In [45]:
tokenizer=tiktoken.get_encoding("gpt2")

In [46]:
tokenizer.encode("Hello world")

[15496, 995]

In [47]:
tokenizer.decode(tokenizer.encode("Hello world"))

'Hello world'

In [73]:
tokenizer.n_vocab

50257

In [48]:
text=(
    "Hello, do you like tea? <|endoftext|> In nahajwid 154adas 545a1s assaxscdsf jhajfkndjhcgasu the sunlit terraces"
    "of someunknownPlace."
)

tokenizer.encode(text,allowed_special={"<|endoftext|>"})

[15496,
 11,
 466,
 345,
 588,
 8887,
 30,
 220,
 50256,
 554,
 299,
 993,
 1228,
 28029,
 24235,
 38768,
 642,
 2231,
 64,
 16,
 82,
 840,
 897,
 1416,
 9310,
 69,
 474,
 71,
 1228,
 69,
 74,
 358,
 73,
 71,
 66,
 22649,
 84,
 262,
 4252,
 18250,
 8812,
 2114,
 1659,
 617,
 34680,
 27271,
 13]

## 2.6 Data sampling with a sliding window（滑动窗口的数据集采样）

In [49]:
with open("the-verdict.txt","r",encoding="utf-8") as f:
    raw_text=f.read()

enc_text=tokenizer.encode(raw_text)
print(len(enc_text))

5145


In [50]:
enc_sample=enc_text[50:]

In [51]:
len(enc_sample)

5095

In [52]:
context_size=4

x=enc_sample[:context_size]
y=enc_sample[1:context_size+1]

print(f"x:{x}")
print(f"y:     {y}")

x:[290, 4920, 2241, 287]
y:     [4920, 2241, 287, 257]


In [53]:
# for i in range(1,context_size+1):
#     context=enc_sample[:i]
#     desired=enc_sample[i]

#     print(context,"----->",desired)

In [54]:
for i in range(1,context_size+1):
    context=enc_sample[:i]
    desired=enc_sample[i]

    print(tokenizer.decode(context),"----->",tokenizer.decode([desired]))

 and ----->  established
 and established ----->  himself
 and established himself ----->  in
 and established himself in ----->  a


In [55]:
import torch

In [56]:
torch.__version__

'2.5.1+cu121'

In [61]:
from torch.utils.data import Dataset,DataLoader

class GPTDatasetV1(Dataset):
    def __init__(self,txt,tokenizer,max_length,stride):
        self.input_ids=[]
        self.target_ids=[]

        #Tokenize the entire text
        token_ids=tokenizer.encode(txt,allowed_special={"<|endoftext|>"})

        #Use a sliding window to chunk the book into overlapping sequences of max_length
        for i in range(0,len(token_ids)-max_length,stride):
            input_chunk=token_ids[i:i+max_length]
            target_chunk=token_ids[i+1:i+max_length+1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self,idx):
        return self.input_ids[idx],self.target_ids[idx]

In [62]:
def create_dataloader_v1(txt,batch_size=4,max_length=256,
                         stride=128,shuffle=True,drop_last=True,
                         num_workers=0):

    # Initialize the tokenizer
    tokenizer=tiktoken.get_encoding("gpt2")

    # Create dataset
    dataset=GPTDatasetV1(txt,tokenizer,max_length,stride)

    # Create dataloader
    dataloader=DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

In [63]:
with open("the-verdict.txt","r",encoding="utf-8") as f:
    raw_text=f.read()

In [66]:
dataloader =create_dataloader_v1(raw_text,batch_size=1,max_length=4,
                         stride=4,shuffle=False
)

data_iter=iter(dataloader)
first_batch=next(data_iter)
print(first_batch)

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


In [69]:
second_batch=next(data_iter)
print(second_batch)

[tensor([[15632,   438,  2016,   257]]), tensor([[ 438, 2016,  257,  922]])]


In [70]:
dataloader =create_dataloader_v1(raw_text,batch_size=8,max_length=4,stride=4,shuffle=False)

data_iter=iter(dataloader)
inputs,targets=next(data_iter)
print("Inputs:\n",inputs)
print("\nTargets:\n",targets)

Inputs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Targets:
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


## 2.7 Creating token embeddings

In [71]:
inputs_ids=torch.tensor([ 2,   3,    5,   1])

In [85]:
vocab_size=6
output_dim=3

# torch.manual_seed(123)
embedding_layer=torch.nn.Embedding(tokenizer.n_vocab,output_dim)

In [86]:
print(embedding_layer.weight)

Parameter containing:
tensor([[-1.0016, -0.0149, -0.3308],
        [-0.4308, -0.6224,  2.0837],
        [ 1.0277,  0.0097,  0.4274],
        ...,
        [ 0.6362,  0.7731, -0.0231],
        [-0.6587, -1.3706, -0.9119],
        [-0.0297, -0.0621,  0.0703]], requires_grad=True)


In [76]:
embedding_layer(torch.tensor([3]))

tensor([[-0.4015,  0.9666, -1.1481]], grad_fn=<EmbeddingBackward0>)

In [77]:
embedding_layer(torch.tensor([2]))

tensor([[ 1.2753, -0.2010, -0.1606]], grad_fn=<EmbeddingBackward0>)

In [78]:
embedding_layer(inputs_ids)

tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)

In [79]:
inputs_ids

tensor([2, 3, 5, 1])

## 2.8 Encoding word positions  

In [87]:
vocab_size=50257
output_dim=256

token_embedding_layer=torch.nn.Embedding(vocab_size,output_dim)

In [88]:
max_length=4
dataloader=create_dataloader_v1(
    raw_text,batch_size=8,max_length=max_length,
    stride=max_length,shuffle=False
)
data_iter=iter(dataloader)
inputs,targets=next(data_iter)

In [89]:
print("Token IDs:\n",inputs)
print("\nInputs shape:\n",inputs.shape)

Token IDs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Inputs shape:
 torch.Size([8, 4])


In [103]:
token_embeddings=token_embedding_layer(inputs)
token_embeddings.shape

torch.Size([8, 4, 256])

In [104]:
context_length=max_length
pos_embedding_layer=torch.nn.Embedding(context_length,output_dim)

In [105]:
torch.arange(max_length)

tensor([0, 1, 2, 3])

In [106]:
pos_embedding_layer.weight

Parameter containing:
tensor([[ 0.8270,  1.2521,  1.2973,  ...,  2.1579,  1.0546, -0.9734],
        [ 0.9186, -0.6091,  0.6546,  ..., -0.0358,  0.5877, -0.2346],
        [ 1.3408, -1.8157, -0.4682,  ...,  0.6272,  0.2101,  1.9348],
        [-0.4677,  0.7937, -0.0586,  ...,  0.9890, -2.3555, -0.2462]],
       requires_grad=True)

In [107]:
pos_embedding_layer(torch.arange(max_length))

tensor([[ 0.8270,  1.2521,  1.2973,  ...,  2.1579,  1.0546, -0.9734],
        [ 0.9186, -0.6091,  0.6546,  ..., -0.0358,  0.5877, -0.2346],
        [ 1.3408, -1.8157, -0.4682,  ...,  0.6272,  0.2101,  1.9348],
        [-0.4677,  0.7937, -0.0586,  ...,  0.9890, -2.3555, -0.2462]],
       grad_fn=<EmbeddingBackward0>)

In [108]:
pos_embeddings=pos_embedding_layer(torch.arange(max_length))
print(pos_embeddings.shape)

torch.Size([4, 256])


In [109]:
token_embeddings.shape

torch.Size([8, 4, 256])

In [110]:
pos_embeddings.shape

torch.Size([4, 256])

In [114]:
input_embeddings=token_embeddings+pos_embeddings
print(input_embeddings.shape)

torch.Size([8, 4, 256])
